In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import nltk
import re
import os
import pickle

# **Read data**

In [2]:
df = pd.read_csv('/kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv')
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


# **Describe data**

In [3]:
df.describe().T

,count,unique,top,freq
review,50000,49582,Loved today's show!!! It was a variety and not...,5
sentiment,50000,2,positive,25000


In [4]:
df['sentiment'].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [5]:
df.replace({"sentiment": {"positive": 1, "negative": 0}}, inplace=True)

/tmp/ipykernel_35/1137712857.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace({"sentiment": {"positive": 1, "negative": 0}}, inplace=True)


# **Preprocess data**

In [6]:
def remove_tags(str):
  #remove HTML tags
  res = re.sub(r'<[^>]+>', '', str)

  #remove URLs
  res = re.sub(r'https?://\S+', '', res)

  #remove non-alphanumeric characters
  res = re.sub(r'[^a-zA-Z0-9' + r'\s]', '', res)

  #convert to lower case
  res = res.lower()
  return res

In [7]:
df['review'] = df['review'].apply(remove_tags)

In [8]:
nltk.download('stopwords')

from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

df['review'] = df['review'].apply(lambda x: ' '.join([word for word in x.split() if word not in (stop_words)]))

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [9]:
#nltk.download('wordnet')
#w_tokenizer = nltk.tokenize.WhitespaceTokenizer()
#lemmatizer = nltk.stem.WordNetLemmatizer()
#def lemmatize_text(text):
#    st = ""
#    for w in w_tokenizer.tokenize(text):
#        st = st + lemmatizer.lemmatize(w) + " "
#    return st
#df['review'] = df.review.apply(lemmatize_text)

In [10]:
# split data into training data and test data
train_data, test_data = train_test_split(df, test_size=0.2, random_state=42, stratify=df['sentiment'])

# **Tokenization**

In [11]:
df.head()

,review,sentiment
0,one reviewers mentioned watching 1 oz episode ...,1
1,wonderful little production filming technique ...,1
2,thought wonderful way spend time hot summer we...,1
3,basically theres family little boy jake thinks...,0
4,petter matteis love time money visually stunni...,1


In [12]:
vectorize = TfidfVectorizer()

X_train = vectorize.fit_transform(train_data["review"])

X_test = vectorize.transform(test_data["review"])

In [13]:
Y_train = train_data["sentiment"]
Y_test = test_data["sentiment"]

# **Train model**

In [14]:
model = SVC()
model.fit(X_train, Y_train)

SVC()

In [18]:
model.score(X_train, Y_train)

0.991825

# **Prediction**

In [19]:
# Validate the model
y_pred = model.predict(X_test)
accuracy = accuracy_score(Y_test, y_pred)
print("Test Accuracy:", accuracy)

Test Accuracy: 0.9031


In [20]:
from sklearn.metrics import classification_report
print(classification_report(Y_test, y_pred))

              precision    recall  f1-score   support

           0       0.91      0.89      0.90      5000
           1       0.90      0.91      0.90      5000

    accuracy                           0.90     10000
   macro avg       0.90      0.90      0.90     10000
weighted avg       0.90      0.90      0.90     10000



In [ ]:
os.makedirs('saved_models', exist_ok=True)

with open('saved_models/svm.pkl', 'wb') as f:
    pickle.dump({'model': model, 'vectorizer': vectorize}, f)